# SoninkeGPT — générer les 20 audios « tête et visage » avec la voix du référent

Ce notebook prépare une génération en lot avec OmniVoice et une **voix de référence fournie par le référent**. Il utilise la transcription de référence exacte `Daga goli. Li taxu.` et le fichier `reference_soninke.wav` déjà présent dans Google Drive, s'il existe.

- Les WAV sont enregistrés dans Google Drive et ne sont pas publiés automatiquement sur GitHub.
- Chaque génération reste en attente de validation humaine.
- Le modèle OmniVoice est documenté avec des poids à licence non commerciale : ne pas l'utiliser pour un service commercial sans autorisation compatible.
- Si aucun GPU gratuit n'est disponible, arrêter sans payer. La génération n'est pas garantie.

In [ ]:
%pip -q install omnivoice soundfile pandas

In [ ]:
from pathlib import Path
from google.colab import drive, files
from IPython.display import Audio, display
import pandas as pd
import numpy as np
import soundfile as sf
import hashlib, json, zipfile, datetime, urllib.request

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/SONINKE_TTS_LIBRE')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_DIR = PROJECT_DIR / 'head_face_batch'
AUDIO_DIR = BATCH_DIR / 'audio'
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

REF_PATH = PROJECT_DIR / 'reference_soninke.wav'
REF_TEXT = 'Daga goli. Li taxu.'

if not REF_PATH.exists():
    print('La référence vocale n’est pas encore dans Google Drive.')
    print('Téléverse reference_soninke.wav, ou les deux originaux Daga goli.wav et Li taxu.wav.')
    uploaded = files.upload()
    for name, data in uploaded.items():
        (PROJECT_DIR / name).write_bytes(data)
    if not REF_PATH.exists():
        a_path = PROJECT_DIR / 'Daga goli.wav'
        b_path = PROJECT_DIR / 'Li taxu.wav'
        if a_path.exists() and b_path.exists():
            a, sr_a = sf.read(a_path)
            b, sr_b = sf.read(b_path)
            if sr_a != sr_b:
                raise ValueError(f'Les fréquences audio diffèrent : {sr_a} Hz et {sr_b} Hz.')
            if a.ndim > 1: a = a.mean(axis=1)
            if b.ndim > 1: b = b.mean(axis=1)
            silence = np.zeros(int(0.4 * sr_a), dtype=np.float32)
            combined = np.concatenate([a.astype(np.float32), silence, b.astype(np.float32)])
            sf.write(REF_PATH, combined, sr_a)
        else:
            raise FileNotFoundError('Référence absente : fournir reference_soninke.wav ou les deux enregistrements originaux.')
print('Référence de voix :', REF_PATH)
print('Transcription de référence :', REF_TEXT)
display(Audio(str(REF_PATH)))

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('ARRÊT SÛR : aucun GPU disponible. Ne paie pas pour continuer; réessaie avec une session gratuite plus tard.')
print('GPU disponible :', torch.cuda.get_device_name(0))
from omnivoice import OmniVoice
model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map='cuda:0', dtype=torch.float16)
lang_names = [str(x).lower() for x in model.supported_language_names()]
if not any('sonink' in x for x in lang_names):
    raise RuntimeError('ARRÊT : cette version du modèle ne confirme pas la langue Soninke.')
SAMPLE_RATE = int(model.sampling_rate)
print('Fréquence du modèle :', SAMPLE_RATE)

In [ ]:
CSV_URL = 'https://raw.githubusercontent.com/aydiarra-star/soninkeGPT/main/data/source/head-face-tts-wordlist.csv'
wordlist_path = BATCH_DIR / 'head-face-tts-wordlist.csv'
with urllib.request.urlopen(CSV_URL, timeout=30) as response:
    wordlist_path.write_bytes(response.read())
df = pd.read_csv(wordlist_path, dtype=str).fillna('')
required = {'id', 'text', 'francais', 'source', 'license', 'attribution'}
if not required.issubset(df.columns):
    raise ValueError(f'Colonnes manquantes : {required - set(df.columns)}')
if df['id'].duplicated().any():
    raise ValueError('Les identifiants du lot doivent être uniques.')
print('Entrées à générer :', len(df))
display(df[['id', 'text', 'francais']])

## Génération des WAV

La génération utilise `ref_audio` et `ref_text` pour guider le modèle avec la voix de référence. Les résultats ne sont pas considérés comme validés linguistiquement par le simple fait qu'un WAV a été créé. Le manifeste est sauvegardé après chaque entrée pour faciliter la reprise.

In [ ]:
MANIFEST = BATCH_DIR / 'manifest.csv'
existing = {}
if MANIFEST.exists():
    old = pd.read_csv(MANIFEST, dtype=str).fillna('')
    existing = {r['id']: r for _, r in old.iterrows()}

records = []
for _, row in df.iterrows():
    item_id, phrase = row['id'].strip(), row['text'].strip()
    wav_path = AUDIO_DIR / f'{item_id}.wav'
    previous = existing.get(item_id, {})
    error = ''
    if wav_path.exists() and wav_path.stat().st_size > 1000:
        status = 'audio_present_reused'
    else:
        status = 'generation_error'
        try:
            print(f"Génération : {item_id} — {phrase}")
            generated = model.generate(
                text=phrase,
                language='soninke',
                ref_audio=str(REF_PATH),
                ref_text=REF_TEXT,
                num_step=32
            )
            if generated is None or len(generated) == 0 or len(generated[0]) == 0:
                raise RuntimeError('Le modèle n’a retourné aucun échantillon audio.')
            samples = np.asarray(generated[0]).squeeze()
            if samples.ndim != 1 or not np.isfinite(samples).all():
                raise RuntimeError('Audio invalide : forme inattendue ou valeurs non finies.')
            sf.write(str(wav_path), samples, SAMPLE_RATE, subtype='PCM_16')
            status = 'generated'
            display(Audio(str(wav_path)))
        except Exception as exc:
            error = f'{type(exc).__name__}: {exc}'
            print('ERREUR — arrêt du lot pour éviter des répétitions coûteuses :', error)
    duration, peak, sha256 = '', '', ''
    if wav_path.exists() and wav_path.stat().st_size > 1000:
        info = sf.info(str(wav_path))
        data, _ = sf.read(str(wav_path), always_2d=True)
        duration = round(float(info.duration), 3)
        peak = round(float(np.max(np.abs(data))) if data.size else 0.0, 5)
        sha256 = hashlib.sha256(wav_path.read_bytes()).hexdigest()
    records.append({
        'id': item_id, 'text': phrase, 'francais': row['francais'],
        'source': row['source'], 'license': row['license'], 'attribution': row['attribution'],
        'audio_file': str(wav_path) if wav_path.exists() else '',
        'technical_status': status, 'duration_seconds': duration,
        'peak_amplitude': peak, 'sha256': sha256, 'generation_error': error,
        'human_pronunciation_validation': previous.get('human_pronunciation_validation', 'pending'),
        'human_notes': previous.get('human_notes', '')
    })
    pd.DataFrame(records).to_csv(MANIFEST, index=False, encoding='utf-8-sig')
    if error:
        break

print('Manifeste :', MANIFEST)
display(pd.DataFrame(records)[['id','text','technical_status','duration_seconds','human_pronunciation_validation']])
print('WAV présents :', sum(1 for r in records if r['audio_file'] and Path(r['audio_file']).exists()), '/', len(df))

In [ ]:
zip_path = BATCH_DIR / 'head-face-audio-review.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    if MANIFEST.exists():
        zf.write(MANIFEST, arcname='manifest.csv')
    for wav_path in sorted(AUDIO_DIR.glob('*.wav')):
        zf.write(wav_path, arcname=f'audio/{wav_path.name}')
print('Archive de contrôle :', zip_path)
print('Les WAV restent dans Google Drive. Avant toute intégration au dépôt, écoute les résultats et confirme la prononciation.')


## Après la génération

1. Écouter les 20 fichiers et vérifier la prononciation, en particulier les variantes `yesso/sinke` et `xanné/xorré`.
2. Si le GPU ou le quota bloque le modèle, conserver le message d'erreur; ne pas déclarer le lot réussi.
3. Télécharger `head-face-audio-review.zip` depuis Google Drive si tu veux me transmettre les WAV et le manifeste pour l'intégration.
4. Les audios ne seront reliés aux fiches du dictionnaire et publiés qu'après vérification des fichiers et confirmation humaine.
